# 05. Pruebas con Granite 4.2

Se comparan **Granite 4.2 de 3B, 8B y 30B** con los casos `C001` a `C008`, en español e inglés (**48 pruebas**).

Cada resultado se agrega a `resultados.csv`.

## 1. Cargar los casos, los modelos y resultados previos

In [10]:
import os
import csv
import json
import time
from copy import deepcopy

import requests

archivo_resultados = "resultados.csv"
url_ollama = "http://localhost:11434/api"

# Parámetros de generación para las pruebas Granite
num_predict = 4096
num_ctx = 8192

columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "solicitud_json", "estado",
    "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

# Cargar casos
with open("casos2.json", "r", encoding="utf-8-sig") as f:
    datos = json.load(f)

casos = {c["id_caso"]: c for c in datos["casos"]}

# Cargar modelos Granite en orden de tamaño (3B, 8B, 30B)
ids_granite = ["M07", "M03", "M21"]
with open("inventario_22_llm.csv", "r", encoding="utf-8-sig", newline="") as f:
    modelos_por_id = {fila["id_modelo"]: fila for fila in csv.DictReader(f)}

modelos_a_probar = [modelos_por_id[i] for i in ids_granite]

# Leer resultados previos o crear el CSV si no existe
if os.path.exists(archivo_resultados):
    with open(archivo_resultados, "r", encoding="utf-8-sig", newline="") as f:
        registros = list(csv.DictReader(f))
else:
    with open(archivo_resultados, "w", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=columnas).writeheader()
    registros = []

# Guardar en un conjunto qué pruebas ya quedaron completas
completadas = set()
for fila in registros:
    if fila["estado"] == "completa":
        completadas.add((fila["id_modelo"], fila["id_caso"], fila["idioma"]))

ids_casos = [f"C{i:03d}" for i in range(1, 9)]
idiomas = ["es", "en"]

for m in modelos_a_probar:
    print(m["id_modelo"], "|", m["nombre_modelo"])
print("Casos:", len(ids_casos), "| Idiomas:", len(idiomas))
print("Solicitudes previstas:", len(modelos_a_probar) * len(ids_casos) * len(idiomas))
print("Registros que ya tiene el CSV:", len(registros))

M07 | Granite 4.2 3B
M03 | Granite 4.2 8B
M21 | Granite 4.2 30B
Casos: 8 | Idiomas: 2
Solicitudes previstas: 48
Registros que ya tiene el CSV: 8


## 2. Preparar la solicitud

El LLM recibe la predicción y las publicaciones de ambos autores. Se permite generar hasta **4096 tokens** (contexto: **8192**).

In [11]:
instrucciones = """Redacta una justificación semántica breve de la predicción de coautoría recibida, con un máximo de 180 palabras.

Responde únicamente en tres apartados:

1. Predicción: indica la clasificación recibida. Si hay probabilidad y umbral, menciona sus valores.
2. Coincidencias y diferencias: explica los temas o métodos de las publicaciones de ambos autores y cita sus artículos.
3. Limitaciones: indica qué información falta o qué afirmaciones no pueden respaldarse.

Reglas:
- La predicción ya fue calculada. No intentes explicar cómo la obtuvo el predictor.
- Utiliza la clasificación conjunta cuando exista; en caso contrario, utiliza la clasificación por umbral indicada para el método.
- Si faltan probabilidades, indícalo sin dejar de comparar las publicaciones.
- La similitud entre artículos no es una probabilidad de coautoría.
- Una similitud indefinida no significa que no existan coincidencias.
- No inventes información ni afirmes que las coincidencias causaron la predicción.
- No repitas las instrucciones ni copies los resúmenes completos."""

def preparar_solicitud(caso, idioma, etiqueta):
    entrada = deepcopy(caso["entrada_llm"])
    umbral_general = datos["regla_coautoria"].get("umbral_definitivo")

    for metodo in entrada["prediccion_coautoria"]["resultados_por_metodo"]:
        p = metodo.get("probabilidad_coautoria")
        umbral = metodo.get("umbral_aplicado")
        if umbral is None:
            umbral = umbral_general
        metodo["umbral_aplicado"] = umbral

        if p is not None and umbral is not None:
            metodo["clasificacion_por_umbral"] = "favorable" if p >= umbral else "desfavorable"
        else:
            metodo["clasificacion_por_umbral"] = None

    instruccion_idioma = datos["idiomas"]["instruccion_por_idioma"][idioma]
    return {
        "model": etiqueta,
        "messages": [
            {"role": "system", "content": instrucciones + "\n" + instruccion_idioma},
            {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
        ],
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "num_ctx": num_ctx, "num_predict": num_predict}
    }

## 3. Consultar a Ollama

Se registra la respuesta, el tiempo y el estado de cada prueba.

In [12]:
def pedir_justificacion(solicitud):
    inicio = time.time()
    enviada = solicitud
    resultado = {"estado": "error", "respuesta": "", "error": ""}

    try:
        r = requests.post(url_ollama + "/chat", json=enviada, timeout=1800)

        # Si el modelo no soporta 'think', reintentar sin ese campo
        if r.status_code == 400 and "think" in r.text.lower():
            enviada = solicitud.copy()
            enviada.pop("think")
            r = requests.post(url_ollama + "/chat", json=enviada, timeout=1800)

        r.raise_for_status()
        salida = r.json()
        texto = salida.get("message", {}).get("content", "")
        resultado["respuesta"] = texto

        if salida.get("done_reason") == "length":
            resultado["estado"] = "truncada"
        elif salida.get("done") and texto.strip() != "":
            resultado["estado"] = "completa"
        else:
            resultado["estado"] = "incompleta"

    except Exception as e:
        resultado["error"] = str(e)

    resultado["tiempo_s"] = round(time.time() - inicio, 2)
    return resultado, enviada

## 4. Ejecutar las 48 pruebas

Se ejecutan los tres modelos uno por uno con los ocho casos en ambos idiomas. Cada respuesta se guarda al terminar en `resultados.csv` y se omiten las que ya estén completas.

In [13]:
# Revisar modelos instalados en Ollama
r = requests.get(url_ollama + "/tags", timeout=15)
r.raise_for_status()
instalados = [m["name"].lower() for m in r.json()["models"]]

faltantes = []
nuevas = 0

for modelo in modelos_a_probar:
    id_modelo = modelo["id_modelo"]
    etiqueta = modelo["etiqueta_ollama"]
    nombre = modelo["nombre_modelo"]

    if etiqueta.lower() not in instalados:
        faltantes.append(etiqueta)
        print("NO INSTALADO:", id_modelo, "|", etiqueta)
        continue

    print(f"\nMODELO: {id_modelo} | {nombre}")

    for id_caso in ids_casos:
        for idioma in idiomas:
            if (id_modelo, id_caso, idioma) in completadas:
                print("  ", id_caso, idioma, "| ya completada")
                continue

            # Contar cuántos intentos previos hay para esta combinación
            intento = 1
            for f in registros:
                if f["id_modelo"] == id_modelo and f["id_caso"] == id_caso and f["idioma"] == idioma:
                    intento += 1

            solicitud = preparar_solicitud(casos[id_caso], idioma, etiqueta)
            resultado, enviada = pedir_justificacion(solicitud)

            fila = {
                "id_caso": id_caso,
                "id_modelo": id_modelo,
                "idioma": idioma,
                "intento": intento,
                "solicitud_json": json.dumps(enviada, ensure_ascii=False),
                "estado": resultado["estado"],
                "tiempo_s": resultado["tiempo_s"],
                "respuesta": resultado["respuesta"],
                "error": resultado["error"],
                "evaluacion": "",
                "motivo_evaluacion": ""
            }

            with open(archivo_resultados, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=columnas).writerow(fila)

            registros.append(fila)
            if fila["estado"] == "completa":
                completadas.add((id_modelo, id_caso, idioma))
            nuevas += 1

            print("  ", id_caso, idioma, "|", fila["estado"], "|", fila["tiempo_s"], "s")
            if fila["error"]:
                print("     Error:", fila["error"][:180])

    # Liberar el modelo de memoria antes de cargar el siguiente
    try:
        requests.post(url_ollama + "/generate", json={"model": etiqueta, "keep_alive": 0}, timeout=60)
    except Exception:
        pass

print("\nIntentos nuevos guardados:", nuevas)
print("Total de filas en resultados.csv:", len(registros))
if faltantes:
    print("\nModelos pendientes de instalar:")
    for etiqueta in faltantes:
        print("  ollama pull", etiqueta)


MODELO: M07 | Granite 4.2 3B
   C001 es | completa | 9.4 s
   C001 en | completa | 6.48 s
   C002 es | completa | 6.81 s
   C002 en | completa | 5.84 s
   C003 es | completa | 5.76 s
   C003 en | completa | 5.59 s
   C004 es | completa | 7.57 s
   C004 en | completa | 5.7 s
   C005 es | completa | 7.31 s
   C005 en | completa | 5.47 s
   C006 es | completa | 6.67 s
   C006 en | completa | 5.66 s
   C007 es | completa | 6.62 s
   C007 en | completa | 7.25 s
   C008 es | completa | 6.51 s
   C008 en | completa | 5.68 s

MODELO: M03 | Granite 4.2 8B
   C001 es | completa | 13.79 s
   C001 en | completa | 8.8 s
   C002 es | completa | 10.09 s
   C002 en | completa | 8.87 s
   C003 es | completa | 9.86 s
   C003 en | completa | 7.0 s
   C004 es | completa | 10.07 s
   C004 en | completa | 7.55 s
   C005 es | completa | 8.43 s
   C005 en | completa | 7.05 s
   C006 es | completa | 8.36 s
   C006 en | completa | 7.8 s
   C007 es | completa | 9.72 s
   C007 en | completa | 8.29 s
   C008 es |

## 5. Revisar el avance

In [14]:
completas = 0
for m in modelos_a_probar:
    for id_caso in ids_casos:
        for idioma in idiomas:
            if (m["id_modelo"], id_caso, idioma) in completadas:
                completas += 1

total = len(modelos_a_probar) * len(ids_casos) * len(idiomas)
print(f"Avance: {completas} de {total} respuestas completas")
print("Pendientes:", total - completas)
print("Resultados guardados en:", archivo_resultados)

Avance: 48 de 48 respuestas completas
Pendientes: 0
Resultados guardados en: resultados.csv
